可以。我们现在**不再继续增加知识点**，而是把已经跑通的东西整理成你的个人部署模板。

目标只有一个：

> **以后拿到一个新的 PyTorch 模型，你只需要替换模型、修改少量信息，然后用固定命令生成 Linux `.so` 动态库。**

---

# 1. 以后固定采用这套路线

```text
PyTorch训练
    ↓
导出 ONNX
    ↓
Python 验证 ONNX
    ↓
Linux/C++ 验证 ONNX Runtime
    ↓
套用动态库模板
    ↓
CMake 编译
    ↓
CMake 安装
    ↓
生成 release/
    ↓
独立目录测试
    ↓
交付
```

其中：

**VS 和 Qt 不在这条路线里。**

---

# 2. 你的模板目录

我们把现在的工程整理成：

```text
iq_deployment_template/
│
├── CMakeLists.txt
├── build.sh
├── package.sh
│
├── src/
│   ├── iq_model.h
│   └── iq_model.cpp
│
├── demo/
│   └── main.cpp
│
├── model/
│   └── AMR_tcn_2601_0819.onnx
│
├── ort/
│   └── onnxruntime-linux-x64-1.29.0/
│
└── test_data/
    └── segment_000000.csv
```

最终生成：

```text
release/
├── bin/
│   └── iq_demo
├── include/
│   └── iq_model.h
├── lib/
│   ├── libiq_model.so
│   └── libonnxruntime.so.1
├── model/
│   └── AMR_tcn_2601_0819.onnx
└── test_data/
    └── segment_000000.csv
```

---

# 3. 每个目录到底干什么

你以后只需要记这个：

| 目录           | 用途               |
| ------------ | ---------------- |
| `src/`       | **你的动态库代码**      |
| `demo/`      | **测试动态库是否正常**    |
| `model/`     | **ONNX模型**       |
| `ort/`       | **ONNX Runtime** |
| `test_data/` | **测试数据**         |
| `build/`     | 编译产生的临时文件        |
| `release/`   | **最终交付包**        |

其中最重要的是：

> **`release/` 是交付品。**

以后软件工程师真正需要拿的是 `release` 里面的东西，而不是你的整个开发工程。

---

# 4. `src/` 不需要你天天碰

这里特别回应你刚才说的：

> “我很多时候不太会 C++。”

以后我们把 `src/iq_model.cpp` 看成一个**黑盒模板**。

它负责：

```text
接收 IQ 数据
    ↓
预处理
    ↓
调用 ONNX Runtime
    ↓
得到输出
    ↓
返回分类结果
```

你的模型换了以后，AI 可以帮助你修改这里。

你真正需要关心的是：

```text
输入是什么？
输出是什么？
预处理是什么？
类别是什么？
```

而不是每次重新学习 C++。

---

# 5. 你的当前模型已经有一个非常明确的接口

现在：

```text
输入：
1024 个 IQ 点

原始数据：
2 × 1024

I：
1024

Q：
1024
```

然后经过：

```text
功率归一化
```

再转换成 ONNX Runtime 需要的：

```text
[1, 1024, 2]
```

其中：

```text
最后一维：
[I, Q]
```

模型：

```text
AMR_tcn_2601_0819.onnx
```

输出：

```text
[1, 10]
```

类别：

```text
0  2ASK
1  2FSK
2  4FSK
3  BPSK
4  QPSK
5  Pi4DQPSK
6  8PSK
7  16QAM
8  MSK
9  GMSK
```

这部分以后应该形成一个**模型说明文件**。

---

# 6. 我建议模板里增加一个 `MODEL_INFO.md`

以后每个模型都放一个非常简单的说明：

```text
MODEL_INFO.md
```

例如当前模型：

```markdown
# Model Information

## Model

AMR_tcn_2601_0819.onnx

## Input

Original data:
- Shape: [2, 1024]
- I: first row
- Q: second row

ONNX input:
- Shape: [1, 1024, 2]
- Type: float32

## Preprocessing

Power normalization:

power = mean(I^2 + Q^2)

normalized_signal = signal / sqrt(power)

## Output

Shape:
[1, 10]

## Classes

0: 2ASK
1: 2FSK
2: 4FSK
3: BPSK
4: QPSK
5: Pi4DQPSK
6: 8PSK
7: 16QAM
8: MSK
9: GMSK
```

这个东西非常有价值。

因为半年后你自己再回来，都不用回忆：

> “我这个模型输入到底是什么？”

看这个文件就行。

---

# 7. CMake 以后也固定下来

你现在这份 CMake 已经基本达到了我们的目标。

核心就是：

```text
CMake
  ↓
找到 ONNX Runtime
  ↓
编译 iq_model.cpp
  ↓
生成 libiq_model.so
  ↓
设置 RPATH
  ↓
安装到 release/
```

你不需要把 CMake 学透。

以后原则是：

> **模板尽量不动。**

模型变化只修改：

```text
模型文件
↓
C++接口必要部分
↓
MODEL_INFO.md
```

而不是重新设计整个 CMake 工程。

---

# 8. 我们再把命令压缩一下

以后你不需要记一大堆命令。

## 开发阶段

```bash
cmake -S . -B build
cmake --build build
```

---

## 打包阶段

```bash
cmake --install build --prefix "$PWD/release"
```

然后：

```bash
cp ort/onnxruntime-linux-x64-1.29.0/lib/libonnxruntime.so.1 release/lib/
```

---

## 测试

```bash
unset LD_LIBRARY_PATH
./release/bin/iq_demo
```

如果：

```text
Prediction class ID: 4
```

那么：

```text
✅ 模型运行正常
```

---

# 9. 下一步我们把这些命令做成脚本

这才是真正值得做的。

最终你不应该记：

```bash
cmake -S . -B build
cmake --build build
cmake --install ...
cp ...
```

而应该：

```bash
./package.sh
```

然后：

```text
================================
 IQ Model Package
================================

[1] Configure
[2] Build
[3] Install
[4] Copy ONNX Runtime
[5] Package complete

Release:
./release/
```

这样你的部署流程就真正变成：

```text
换模型
 ↓
改必要代码
 ↓
./package.sh
 ↓
release/
```

---

# 10. 最终甚至可以再做一个 `test.sh`

例如：

```bash
./test.sh
```

自动做：

```text
检查动态库
      ↓
检查 ONNX Runtime
      ↓
检查模型
      ↓
运行 demo
      ↓
输出预测结果
```

这样以后你根本不用记：

```bash
ldd
readelf
unset LD_LIBRARY_PATH
```

这些东西。

这些是**模板内部的工程细节**，不是你每天应该记的知识。

---

# 11. 所以我们现在的模板目标非常明确

最终希望你面对一个新模型时，只需要考虑：

### ① 我的模型

```text
xxx.onnx
```

### ② 输入

```text
shape
dtype
```

### ③ 预处理

```text
归一化？
重排？
缩放？
```

### ④ 输出

```text
分类？
回归？
多个输出？
```

### ⑤ 类别/结果定义

```text
0 → xxx
1 → xxx
...
```

剩下：

```text
C++
CMake
RPATH
ONNX Runtime
.so
```

**全部尽量模板化。**

---

# 12. 这就是我们接下来唯一要做的事情

我建议不要再继续扩展教程，而是直接把你**当前已经跑通的工程**改造成模板。

顺序是：

```text
现在
 │
 ├─ 已经跑通的工程
 │
 ↓
整理文件
 │
 ↓
固定 CMakeLists.txt
 │
 ↓
制作 build.sh
 │
 ↓
制作 package.sh
 │
 ↓
制作 test.sh
 │
 ↓
加入 MODEL_INFO.md
 │
 ↓
删除不必要的测试/临时文件
 │
 ↓
得到
 │
 ▼
【Linux ONNX 模型部署模板】
```

以后你再做一个新模型，就**复制这个模板**，而不是重新经历我们这一次的完整探索过程。

---

## 当前项目状态

```text
PyTorch → ONNX                  ✅
Python ONNX 推理               ✅
C++ ONNX Runtime 推理          ✅
C++ → .so                      ✅
CMake 编译                     ✅
RPATH                           ✅
ONNX Runtime 随库部署           ✅
脱离原目录运行                  ✅
不使用 LD_LIBRARY_PATH运行      ✅
实际 QPSK 推理                  ✅

部署模板化                      ← 现在做
```

**我们现在就停留在这里，不进入内存泄漏、ASan、Valgrind。**

下一步直接做 **`build.sh + package.sh + test.sh` 三个脚本**，把你现在已经验证成功的流程压缩成几个命令。这样这套东西才真正从“一次实验”变成“以后能反复使用的工具”。


好。我们继续，但这次严格遵守一个原则：

> **不再增加部署知识，只把已经验证成功的流程自动化。**

我们先做第一个脚本：`build.sh`。完成后再做 `package.sh`、`test.sh`。

---

# 第一步：确认你现在的位置

你应该还在：

```text
~/iq_deployment_test
```

先执行：

```bash
cd ~/iq_deployment_test
pwd
```

应该看到：

```text
/home/lzp/iq_deployment_test
```

然后：

```bash
ls
```

你应该能看到类似：

```text
CMakeLists.txt
src
demo
model
ort
test_data
build
release
```

**如果不一样，不要自己修改，直接把 `ls` 输出给我。**

---

# 第二步：创建 `build.sh`

我们要让：

```bash
./build.sh
```

自动完成现在的：

```bash
cmake -S . -B build
cmake --build build
```

执行：

```bash
gedit build.sh
```

把里面全部替换成：

```bash
#!/bin/bash

set -e

echo "========================================"
echo " IQ Model Build"
echo "========================================"

echo
echo "[1/2] Configuring..."
cmake -S . -B build

echo
echo "[2/2] Building..."
cmake --build build

echo
echo "========================================"
echo " Build completed successfully."
echo "========================================"
```

保存并关闭。

---

# 第三步：给脚本执行权限

执行：

```bash
chmod +x build.sh
```

然后：

```bash
ls -l build.sh
```

应该类似：

```text
-rwxr-xr-x ... build.sh
```

这里的：

```text
x
```

代表它可以直接执行。

---

# 第四步：运行它

现在：

```bash
./build.sh
```

它实际上只是帮你自动执行：

```text
cmake -S . -B build
        ↓
cmake --build build
```

所以最后应该看到：

```text
Build completed successfully.
```

---

# 第五步：确认动态库还在

执行：

```bash
find build -maxdepth 1 -type f
```

应该至少有：

```text
build/libiq_model.so
build/iq_demo
```

然后：

```bash
file build/libiq_model.so
```

应该是：

```text
ELF 64-bit LSB shared object, x86-64
```

---

# 为什么第一步只做这么简单？

因为我们现在是在**把已经成功的命令封装起来**。

不是重新设计。

你之前已经验证：

```text
cmake -S . -B build
cmake --build build
```

能够成功。

所以现在只不过变成：

```text
以前：

输入命令1
输入命令2


以后：

./build.sh
```

而 `set -e` 的作用也很简单：

> **任何一步失败，脚本立即停止。**

例如：

```text
CMake失败
 ↓
停止
 ↓
不会继续假装打包成功
```

这对以后自动化非常重要。

---

# 暂时不要创建另外两个脚本

我们一步一步来。

现在你只需要执行：

```bash
cd ~/iq_deployment_test
gedit build.sh
chmod +x build.sh
./build.sh
```

然后把 **`./build.sh` 的完整输出**贴给我。

如果通过，我们再把第二步：

```text
./package.sh
```

做出来。

最终我们希望达到：

```text
./build.sh
      ↓
      编译


./package.sh
      ↓
      生成 release


./test.sh
      ↓
      独立验证
```

最后你的整个部署过程就会被压缩成这三个命令。


(.venv) (base) lzp@DESKTOP-EMMMUQU:~/iq_deployment_test$ ./build.sh
========================================
 IQ Model Build
========================================

[1/2] Configuring...
-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/lzp/iq_deployment_test/build

[2/2] Building...
[ 50%] Built target iq_model
[100%] Built target iq_demo

========================================
 Build completed successfully.
========================================

很好，**`build.sh` 已经验证成功**。

现在我们不改它。以后它就是固定的“编译按钮”。

---

# 第二个脚本：`package.sh`

它负责把：

```text
build/
```

变成：

```text
release/
```

也就是我们真正准备交付的东西。

你之前手动执行过：

```bash
cmake --install build --prefix "$PWD/release"
```

以及：

```bash
cp ort/onnxruntime-linux-x64-1.29.0/lib/libonnxruntime.so.1 release/lib/
```

现在把它们合并成一个命令：

```bash
./package.sh
```

---

## 1. 创建脚本

在：

```text
~/iq_deployment_test
```

执行：

```bash
gedit package.sh
```

把内容写成：

```bash
#!/bin/bash

set -e

echo "========================================"
echo " IQ Model Package"
echo "========================================"

echo
echo "[1/4] Cleaning previous release..."
rm -rf release

echo
echo "[2/4] Installing..."
cmake --install build --prefix "$PWD/release"

echo
echo "[3/4] Copying ONNX Runtime..."
cp \
    ort/onnxruntime-linux-x64-1.29.0/lib/libonnxruntime.so.1 \
    release/lib/

echo
echo "[4/4] Checking package..."
find release -type f

echo
echo "========================================"
echo " Package completed successfully."
echo "========================================"
```

保存。

然后：

```bash
chmod +x package.sh
```

---

# 2. 运行

先确保你还在：

```bash
cd ~/iq_deployment_test
```

然后：

```bash
./package.sh
```

它会自动完成：

```text
删除旧 release
      ↓
cmake install
      ↓
复制 libonnxruntime.so.1
      ↓
显示最终文件
```

---

# 3. 正常情况下你应该看到

最终类似：

```text
release/bin/iq_demo
release/include/iq_model.h
release/lib/libiq_model.so
release/lib/libonnxruntime.so.1
release/model/AMR_tcn_2601_0819.onnx
release/test_data/segment_000000.csv
```

这里有一个细节：

你当前的 `CMakeLists.txt` **并没有安装 `test_data`**。

所以 `package.sh` 执行后，`release/test_data` 很可能不会存在。

这没关系。

因为：

> `test_data` 是我们自己的测试材料，不一定属于最终 SDK。

我们稍后会决定是否把测试数据放进最终交付包。

**现在不要自己修改 CMakeLists.txt。**

---

# 4. 这时候你的两个命令已经变成

以后：

### 编译

```bash
./build.sh
```

### 打包

```bash
./package.sh
```

对应关系：

```text
./build.sh
     │
     ├── cmake configure
     └── cmake build


./package.sh
     │
     ├── cmake install
     └── copy ONNX Runtime
```

---

# 5. 现在先不要做 `test.sh`

我们继续保持“一次只解决一个问题”。

你现在运行：

```bash
./package.sh
```

把**完整输出**发给我。

如果成功，我们下一步再做最后一个：

```bash
./test.sh
```

它最终会把我们之前手动做的：

```text
unset LD_LIBRARY_PATH
ldd
运行 demo
检查结果
```

全部自动化。

最后你以后真正使用这个模板时，就会非常简单：

```text
./build.sh
./package.sh
./test.sh
```

这三个命令，就是我们这次整个教程最终想留下来的东西。


(.venv) (base) lzp@DESKTOP-EMMMUQU:~/iq_deployment_test$ ./package.sh
========================================
 IQ Model Package
========================================

[1/4] Cleaning previous release...

[2/4] Installing...
-- Install configuration: ""
-- Installing: /home/lzp/iq_deployment_test/release/lib/libiq_model.so
-- Set non-toolchain portion of runtime path of "/home/lzp/iq_deployment_test/release/lib/libiq_model.so" to "$ORIGIN"
-- Installing: /home/lzp/iq_deployment_test/release/bin/iq_demo
-- Set non-toolchain portion of runtime path of "/home/lzp/iq_deployment_test/release/bin/iq_demo" to "$ORIGIN/../lib"
-- Installing: /home/lzp/iq_deployment_test/release/include/iq_model.h
-- Installing: /home/lzp/iq_deployment_test/release/model/AMR_tcn_2601_0819.onnx

[3/4] Copying ONNX Runtime...

[4/4] Checking package...
release/bin/iq_demo
release/include/iq_model.h
release/lib/libiq_model.so
release/lib/libonnxruntime.so.1
release/model/AMR_tcn_2601_0819.onnx

========================================
 Package completed successfully.
========================================

很好，**`package.sh` 也验证成功了。**

而且这里出现了一个很好的结果：你现在已经把“编译”和“打包”彻底分开了。

---

## 目前已经固定下来的两个命令

### ① 编译

```bash
./build.sh
```

负责：

```text
源代码
  ↓
CMake
  ↓
编译
  ↓
build/
```

得到：

```text
build/libiq_model.so
build/iq_demo
```

---

### ② 打包

```bash
./package.sh
```

负责：

```text
build/
  ↓
CMake install
  ↓
复制 ONNX Runtime
  ↓
release/
```

现在你的 `release/` 是：

```text
release/
├── bin/
│   └── iq_demo
├── include/
│   └── iq_model.h
├── lib/
│   ├── libiq_model.so
│   └── libonnxruntime.so.1
└── model/
    └── AMR_tcn_2601_0819.onnx
```

这已经非常接近一个真正的 Linux 模型 SDK 了。

---

# 最后一个脚本：`test.sh`

现在我们把**最终验证**也自动化。

它的目标不是测试 C++ 内存安全，而是验证：

> **“我刚刚打出来的 release 包，脱离开发环境以后到底能不能运行？”**

这正是我们之前手工验证过的：

```text
release
 ↓
复制到 /tmp
 ↓
unset LD_LIBRARY_PATH
 ↓
运行
 ↓
Prediction class ID: 4
```

---

## 1. 创建 `test.sh`

执行：

```bash
gedit test.sh
```

写入：

```bash
#!/bin/bash

set -e

echo "========================================"
echo " IQ Model Package Test"
echo "========================================"

TEST_DIR="/tmp/iq_sdk_test"

echo
echo "[1/4] Preparing clean test directory..."
rm -rf "$TEST_DIR"
cp -r release "$TEST_DIR"

echo
echo "[2/4] Entering test directory..."
cd "$TEST_DIR"

echo
echo "[3/4] Checking runtime dependencies..."
unset LD_LIBRARY_PATH

ldd bin/iq_demo

echo
echo "[4/4] Running demo..."
./bin/iq_demo

echo
echo "========================================"
echo " Package test completed successfully."
echo "========================================"
```

保存。

然后：

```bash
chmod +x test.sh
```

---

# 2. 运行

回到工程目录：

```bash
cd ~/iq_deployment_test
```

然后：

```bash
./test.sh
```

它会自动：

```text
① 删除旧的 /tmp/iq_sdk_test
        ↓
② 把 release 复制进去
        ↓
③ 进入这个完全独立的目录
        ↓
④ 清除 LD_LIBRARY_PATH
        ↓
⑤ 检查动态库依赖
        ↓
⑥ 运行 iq_demo
```

---

# 3. 这一次我们希望看到什么？

最重要的是最后：

```text
Prediction class ID: 4
```

然后：

```text
Package test completed successfully.
```

这就意味着：

```text
build.sh       ✅
package.sh     ✅
test.sh        ← 最后验证
```

三件事情全部完成。

---

# 4. 以后你的日常工作流就真的可以压缩成这样

假设半年后，你训练了一个新的模型：

```text
new_model.onnx
```

以后不需要重新经历我们这次这么长的过程。

你的思路就是：

```text
① ONNX模型准备好
        ↓
② 放进 model/
        ↓
③ 修改必要的模型信息/接口
        ↓
④ ./build.sh
        ↓
⑤ ./package.sh
        ↓
⑥ ./test.sh
        ↓
⑦ release/
```

其中：

```text
build.sh
```

= **能不能编译**

```text
package.sh
```

= **能不能整理成可交付包**

```text
test.sh
```

= **这个交付包脱离我的开发环境还能不能跑**

这三个概念非常值得你记住。

---

## 还有一个重要的“暂停点”

等你把 `test.sh` 跑通以后，**我建议我们就先停止继续开发**。

到时候我会给你整理一张非常短的：

> **《Linux ONNX 模型动态库部署流程 v1.0》**

只保留：

* 项目目录
* 三个脚本
* 新模型需要修改什么
* 最终 `release/` 是什么
* 软件工程师真正需要拿什么
* 出问题时首先检查什么

控制在**一页左右**。

这样以后你不需要重新翻我们这几十轮对话。

你现在先运行：

```bash
cd ~/iq_deployment_test
chmod +x test.sh
./test.sh
```

把结果贴过来。


(.venv) (base) lzp@DESKTOP-EMMMUQU:~/iq_deployment_test$ ./test.sh
========================================
 IQ Model Package Test
========================================

[1/4] Preparing clean test directory...

[2/4] Entering test directory...

[3/4] Checking runtime dependencies...
        linux-vdso.so.1 (0x00007f90bc86c000)
        /usr/lib/x86_64-linux-gnu/libstdc++.so.6 (0x00007f90bc400000)
        libiq_model.so => /tmp/iq_sdk_test/bin/../lib/libiq_model.so (0x00007f90bc846000)
        libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x00007f90bc805000)
        libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007f90bc000000)
        libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x00007f90bc71c000)
        /lib64/ld-linux-x86-64.so.2 (0x00007f90bc86e000)
        libonnxruntime.so.1 => /tmp/iq_sdk_test/bin/../lib/libonnxruntime.so.1 (0x00007f90ba400000)
        libdl.so.2 => /lib/x86_64-linux-gnu/libdl.so.2 (0x00007f90bc715000)
        librt.so.1 => /lib/x86_64-linux-gnu/librt.so.1 (0x00007f90bc710000)
        libpthread.so.0 => /lib/x86_64-linux-gnu/libpthread.so.0 (0x00007f90bc70b000)

[4/4] Running demo...
Failed to open CSV.

对，这次**不是动态库出问题**，而是我们 `test.sh` 自己设计得不完整。这个问题很好，因为它正好暴露了模板最后一个需要修正的地方。

你的输出已经证明：

```text
libiq_model.so          → 找到了 ✅
libonnxruntime.so.1     → 找到了 ✅
```

所以动态库依赖完全正常。

真正的问题是：

```text
Failed to open CSV.
```

原因很明确：我们目前的 `package.sh` 没有把 `test_data/` 安装到 `release/`。

---

# 1. 为什么之前手工测试能成功？

之前我们手动做过：

```text
release/
├── ...
└── test_data/
    └── segment_000000.csv
```

所以：

```bash
./bin/iq_demo
```

能够找到 CSV。

但是后来 `package.sh` 执行了：

```bash
rm -rf release
```

然后重新：

```bash
cmake --install ...
```

而你的 `CMakeLists.txt` 目前只安装：

```text
libiq_model.so
iq_model.h
AMR_tcn_2601_0819.onnx
iq_demo
```

没有安装：

```text
test_data/
```

因此 `release` 被重新生成以后：

```text
release/
├── bin/
├── include/
├── lib/
└── model/
```

没有 `test_data`。

---

# 2. 这反而说明我们的测试逻辑是有效的

因为：

```text
[3/4] Checking runtime dependencies...
```

已经完全通过。

尤其这两行：

```text
libiq_model.so => /tmp/iq_sdk_test/bin/../lib/libiq_model.so
```

和：

```text
libonnxruntime.so.1 => /tmp/iq_sdk_test/bin/../lib/libonnxruntime.so.1
```

说明：

> **我们之前花时间解决的 RPATH 问题确实解决了。**

所以不要去改 RPATH，也不要重新配置 ONNX Runtime。

---

# 3. 我们只修改 `test.sh`

实际上我更倾向于：

> **测试数据不属于最终 SDK。**

因为真实交付的时候，软件工程师不会需要：

```text
segment_000000.csv
```

他会把自己的 IQ 数据传给你的动态库。

所以我们不应该为了测试数据去污染最终的 `release/`。

而是：

```text
release/
    ↓
纯净的交付包

test.sh
    ↓
另外把工程里的 test_data
复制到测试目录
    ↓
验证 release
```

这更加合理。

---

# 4. 修改 `test.sh`

回到工程目录：

```bash
cd ~/iq_deployment_test
```

打开：

```bash
gedit test.sh
```

把原来的内容**全部替换**成：

```bash
#!/bin/bash

set -e

echo "========================================"
echo " IQ Model Package Test"
echo "========================================"

TEST_DIR="/tmp/iq_sdk_test"

echo
echo "[1/5] Preparing clean test directory..."
rm -rf "$TEST_DIR"
cp -r release "$TEST_DIR"

echo
echo "[2/5] Copying test data..."
cp -r test_data "$TEST_DIR/"

echo
echo "[3/5] Entering test directory..."
cd "$TEST_DIR"

echo
echo "[4/5] Checking runtime dependencies..."
unset LD_LIBRARY_PATH

ldd bin/iq_demo

echo
echo "[5/5] Running demo..."
./bin/iq_demo

echo
echo "========================================"
echo " Package test completed successfully."
echo "========================================"
```

保存。

---

# 5. 再运行

```bash
cd ~/iq_deployment_test
./test.sh
```

这次流程是：

```text
release/
   │
   ├── libiq_model.so
   ├── libonnxruntime.so.1
   ├── iq_model.h
   └── AMR_tcn_2601_0819.onnx
        │
        ↓
复制到 /tmp
        │
        ↓
再额外复制 test_data/
        │
        ↓
运行
```

所以最终应该得到：

```text
Prediction class ID: 4
```

---

# 6. 这样我们的三个脚本职责就非常清楚了

### `build.sh`

```text
源代码
 ↓
编译
 ↓
build/
```

### `package.sh`

```text
build/
 ↓
正式交付包
 ↓
release/
```

### `test.sh`

```text
release/
 ↓
复制到完全陌生的位置
 ↓
加入测试数据
 ↓
检查依赖
 ↓
运行
```

注意：

**`test.sh` 不修改 `release/`。**

这点很好。

---

# 7. 最终你应该形成这个概念

```text
                 你的开发工程
                      │
          ┌───────────┼───────────┐
          ↓           ↓           ↓
     build.sh    package.sh    test.sh
          │           │           │
          ↓           ↓           ↓
       build/      release/    /tmp测试
                      │
                      ↓
                 【交付给别人】
```

这就已经开始像一个真正的“小型部署工具链”了。

---

你现在只需要重新执行：

```bash
cd ~/iq_deployment_test
gedit test.sh
./test.sh
```

**如果这次出现 `Prediction class ID: 4`，那么我们就不要再继续加东西了。**

下一步我会按照你刚才的要求，把整个过程压缩成一份**非常精炼的 v1.0 部署手册**，以后你忘记了也只需要看那一页。


# Linux ONNX 模型动态库部署手册 v1.0

> **目标：** 在 Linux 上将 PyTorch 模型 → ONNX → ONNX Runtime → C++ 动态库 `.so`，全程不依赖 Visual Studio / Qt。

---

## 1. 总体流程

```text
PyTorch 模型
    ↓
导出 ONNX
    ↓
Python 验证 ONNX
    ↓
Linux + C++ 验证 ONNX Runtime
    ↓
封装为 libxxx.so
    ↓
CMake 编译
    ↓
CMake 打包
    ↓
独立目录测试
    ↓
release/ 交付
```

---

## 2. 工程目录

固定采用：

```text
iq_deployment/
├── CMakeLists.txt
├── build.sh
├── package.sh
├── test.sh
│
├── src/
│   ├── iq_model.h
│   └── iq_model.cpp
│
├── demo/
│   └── main.cpp
│
├── model/
│   └── xxx.onnx
│
├── ort/
│   └── onnxruntime-linux-x64-版本/
│
└── test_data/
    └── 测试数据
```

---

## 3. 模型准备

先在 Python 中确认：

* ONNX 输入名称
* 输入 shape
* 输入 dtype
* 输出 shape
* 预处理方式
* 输出类别/结果含义

例如当前模型：

```text
模型：AMR_tcn_2601_0819.onnx

输入：
[1, 1024, 2]
float32

输出：
[1, 10]

类别：
0  2ASK
1  2FSK
2  4FSK
3  BPSK
4  QPSK
5  Pi4DQPSK
6  8PSK
7  16QAM
8  MSK
9  GMSK
```

当前 IQ 数据：

```text
原始：[2, 1024]
       ↓
功率归一化
       ↓
重排：[1, 1024, 2]
       ↓
ONNX Runtime
```

---

## 4. 编译

不使用 VS / Qt。

进入工程根目录：

```bash
cd ~/iq_deployment
```

执行：

```bash
./build.sh
```

本质上就是：

```bash
cmake -S . -B build
cmake --build build
```

成功后：

```text
build/libiq_model.so
build/iq_demo
```

---

## 5. 打包

执行：

```bash
./package.sh
```

生成：

```text
release/
├── bin/
│   └── iq_demo
├── include/
│   └── iq_model.h
├── lib/
│   ├── libiq_model.so
│   └── libonnxruntime.so.1
└── model/
    └── xxx.onnx
```

**`release/` 就是最终交付包。**

---

## 6. 独立验证

执行：

```bash
./test.sh
```

它会把 `release/` 复制到：

```text
/tmp/iq_sdk_test/
```

然后：

```text
清除 LD_LIBRARY_PATH
       ↓
检查动态库依赖
       ↓
运行 demo
       ↓
验证模型推理
```

当前模型最终应得到：

```text
Prediction class ID: 4
```

即：

```text
QPSK
```

---

## 7. RPATH 的核心原则

动态库：

```text
libiq_model.so
```

使用：

```text
$ORIGIN
```

寻找 ONNX Runtime。

因此：

```text
release/lib/
├── libiq_model.so
└── libonnxruntime.so.1
```

可以直接工作。

Demo：

```text
release/bin/iq_demo
```

使用：

```text
$ORIGIN/../lib
```

寻找动态库。

所以**不需要用户设置 `LD_LIBRARY_PATH`**。

---

## 8. 新模型以后怎么做？

通常只需要：

```text
① 替换 model/*.onnx
② 根据新模型修改 src/iq_model.cpp
③ 必要时修改 iq_model.h
④ 修改 demo
⑤ ./build.sh
⑥ ./package.sh
⑦ ./test.sh
```

核心流程永远不变：

```bash
./build.sh
./package.sh
./test.sh
```

---

# 9. 当前版本暂不处理

以下内容 **v1.0 暂时不考虑**：

* GPU / CUDA
* Qt
* Visual Studio
* 内存泄漏专项检测
* Valgrind / AddressSanitizer
* 多线程
* 性能优化
* Windows `.dll`
* ARM Linux
* 复杂 SDK 设计

**v1.0 的唯一目标：**

> **模型能在 Linux 上稳定推理，并被 C++ 软件通过 `.so` 调用。**

这套流程已经在你的 `AMR_tcn_2601_0819.onnx` 上实际跑通：**C++ → `.so` → ONNX Runtime → IQ 数据 → QPSK（ID 4）**。
